We are implementing discrete delta hedging on European options using fixed + proportional transaction costs(based on order size).
We are implementing no trade band hedging(i.e. when delta is within a specified range we wont reblance) and partial hedging (i.e. when delta moves out of the predetermined range we hedge part of the required amount)

In [ ]:
import numpy as np
import scipy.stats as stat #for cdf

In [ ]:
S0 = 100        # initial stock price
K = 100         # strike price
T = 1.0         # maturity in years(time after which option expires)
r = 0.05        # risk-free interest rate
sigma = 0.2     # volatility
N = 252         # number of hedging steps
dt = T/N        # len of each hedging interval

# Transaction costs
fixcost = 0.5        # per trade
propcost = 0.001     # proportional cost based on ordr size

# Hedging parameters
no_trade_band = 0.02    # delta threshold
partial = 0.55   # fraction of hedge adjustment

In [ ]:
#Black scholes delta

def bs_delta(S, K, T, r, sigma):
    if T <= 0:
        return 1.0 if S > K else 0.0
    d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    return stat.norm.cdf(d1)

d1 is the standard measure of where stock price lies relative to strike price after accounting for time to maturity,volatility, interest rate.

Since log stock returns are normally distributed standard normal is followed.

The delta of european options is the cdf of d1 which serves are hedge ratio.

In [ ]:
#simulating price using geometric brownian motion
def sim_price(S0, r, sigma, T, N):
    dt = T / N
    Z = np.random.randn(N)
    S = np.zeros(N + 1)
    S[0] = S0
    for t in range(N):
        S[t+1] = S[t] * np.exp(
            (r - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * Z[t]
        )
    return S


In [ ]:
def hedge_option(S_path,strategy="full"):

    position = 0.0   #number of shares held
    cash = 0.0  #to evaluate strategies independent of initial inestment
    tot_cost = 0.0

    for t in range(len(S_path) - 1):
        tau = T - t * dt
        S = S_path[t]

        target_delta = bs_delta(S, K, tau, r, sigma) #desired hedge acc to black scholes
        delta_change = target_delta - position

        trade = False

        if strategy == "full":
            trade_size = delta_change
            trade = True

        elif strategy == "no_trade_band":
            if abs(delta_change) > no_trade_band:
                trade_size = delta_change
                trade = True

        elif strategy == "partial":
            if abs(delta_change) > no_trade_band:
                trade_size = partial * delta_change
                trade = True

        if trade:
            trade_value = trade_size * S
            cost = fixcost + propcost * abs(trade_value)
            cash -= trade_value + cost
            tot_cost += cost
            position += trade_size

    payoff = max(S_path[-1] - K, 0)
    portfolio_value = position * S_path[-1] + cash

    hedging_error = portfolio_value - payoff
    return hedging_error, tot_cost


The objective value measures the overall cost of the three hedging strategies.

It combines hedging error and transaction costs.

A lower objective value means the strategy achieves a better balance between minimising risk and reducing trading expenses.

In [ ]:
def objective(strategy, paths=1000, lam=1.0):
    errors = []
    costs = []

    for _ in range(paths):
        S_path = sim_price(S0, r, sigma, T, N)
        err, cost = hedge_option(S_path, strategy)
        errors.append(err**2)
        costs.append(cost)
#lam = cost relative to hedging error
    return np.mean(errors) + lam * np.mean(costs)


In [ ]:
strategies = ["full", "no_trade_band", "partial"]

for strat in strategies:
    obj = objective(strat)
    print(f"{strat:15s} Objective Value: {obj:.4f}")


full            Objective Value: 18154.3779
no_trade_band   Objective Value: 3558.4805
partial         Objective Value: 4428.7370


 In this Simulation with transaction costs, full hedging incurs excessive costs despite minimal hedging errors, resulting in the highest objective value. No-trade band hedging achieves the lowest objective by reducing trading frequency, while partial hedging offers a compromise with slightly higher objective than no-trade-band. These results illustrate that in realistic markets, strategies that balance hedging precision and transaction costs outperform continuous full hedging.